# Multimodal Classification of Disaster Data with GNN
## Paper Reproduction Notebook

**Paper:** *Multimodal Classification of Social Media Disaster Posts With Graph Neural Networks and Few-Shot Learning*  
**Authors:** José Nascimento, Paolo Bestagini, Anderson Rocha  
**Repository:** [jdnascim/mm-class-for-disaster-data-with-gnn](https://github.com/jdnascim/mm-class-for-disaster-data-with-gnn)

This notebook reproduces the paper's experiments by:
1. Cloning the **original repository** and using its code as-is
2. Running `feature_fusion.py` via subprocess (matching `exp_5.sh`)
3. Caching features (MaxViT + MPNet) to avoid re-extraction
4. Running all 4 labeled-set sizes × 10 splits = **40 experiments**
5. Aggregating results and comparing with paper's Table V

### Paper Results (Table V — F1-Score %)
| Config | 50 | 100 | 250 | 500 |
|---|---|---|---|---|
| **Ours (GNN Late)** | **74.8±2.0** | **76.3±0.7** | **77.1±0.4** | **—** |

## 1. Setup — Clone Repo & Install Dependencies

In [ ]:
import os, sys, time

# ═══════════════════════════════════════════════════════
# 1a. Clone the original repository
# ═══════════════════════════════════════════════════════
REPO = 'mm-class-for-disaster-data-with-gnn'
REPO_URL = f'https://github.com/jdnascim/{REPO}.git'

if not os.path.isdir(REPO):
    !git clone --depth 1 {REPO_URL}
    print(f'Cloned {REPO}')
else:
    print(f'Repo already exists')

%cd {REPO}
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ═══════════════════════════════════════════════════════
# AUTO-LOGGER: Captures all stages for reporting
# ═══════════════════════════════════════════════════════
import datetime, json as _json

class ExperimentLogger:
    def __init__(self):
        self.start_time = datetime.datetime.now()
        self.stages = []
        self.metadata = {}

    def log_stage(self, name, data):
        entry = {
            'stage': name,
            'timestamp': datetime.datetime.now().isoformat(),
            'elapsed_s': (datetime.datetime.now() - self.start_time).total_seconds(),
            **data
        }
        self.stages.append(entry)
        self._auto_save()
        print(f'[LOG] {name}: {data}')

    def set_meta(self, key, value):
        self.metadata[key] = value

    def _auto_save(self):
        try:
            self.save(f'experiment_log_latest.json')
        except: pass

    def save(self, path='experiment_report.json'):
        report = {
            'metadata': self.metadata,
            'start_time': self.start_time.isoformat(),
            'end_time': datetime.datetime.now().isoformat(),
            'total_time_s': (datetime.datetime.now() - self.start_time).total_seconds(),
            'stages': self.stages
        }
        with open(path, 'w') as f:
            _json.dump(report, f, indent=2, ensure_ascii=False, default=str)
        print(f'Report saved: {path}')
        return report

    def to_markdown(self):
        lines = []
        lines.append(f'# Experiment Report')
        lines.append(f'**Mode:** {self.metadata.get("experiment_mode", "N/A")}')
        lines.append(f'**Start:** {self.start_time.strftime("%Y-%m-%d %H:%M:%S")}')
        lines.append(f'**Total time:** {(datetime.datetime.now()-self.start_time).total_seconds()/60:.1f} min')
        lines.append('')
        for s in self.stages:
            lines.append(f'## {s["stage"]}')
            lines.append(f'*Elapsed: {s["elapsed_s"]/60:.1f} min*')
            for k, v in s.items():
                if k not in ('stage', 'timestamp', 'elapsed_s'):
                    lines.append(f'- **{k}:** {v}')
            lines.append('')
        return '\n'.join(lines)

LOG = ExperimentLogger()
print('Experiment logger initialized')


In [ ]:
# ═══════════════════════════════════════════════════════
# 1b. Verify GPU & PyTorch
# ═══════════════════════════════════════════════════════
import torch
print(f'PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    torch.backends.cudnn.benchmark = True
    print('cuDNN benchmark enabled')
else:
    print('WARNING: No GPU detected! Experiments will be very slow.')
# -- Log GPU info --
if torch.cuda.is_available():
    LOG.log_stage('GPU Setup', {
        'gpu_name': torch.cuda.get_device_name(0),
        'vram_gb': round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
        'pytorch': torch.__version__,
        'cuda': torch.version.cuda
    })


In [ ]:
# ===============================================================
# 1c. Install dependencies (optimized for Colab)
# ===============================================================

t0 = time.time()
import torch
print(f'PyTorch {torch.__version__} | CUDA {torch.version.cuda}')

# PyG (2.4+ works WITHOUT torch-scatter/torch-sparse on PyTorch 2.x)
!pip install -q torch-geometric
print(f'[1/2] PyG ({time.time()-t0:.0f}s)')

# All other deps in one call
t1 = time.time()
!pip install -q numpy pandas scikit-learn tqdm matplotlib pillow scikit-image requests umap-learn timm sentence-transformers jsonlines psutil emoji==1.7.0
print(f'[2/2] Core + ML deps ({time.time()-t1:.0f}s)')

print(f'\nAll dependencies installed! Total: {time.time()-t0:.0f}s')

# -- Log dependencies --
import torch_geometric
LOG.log_stage('Dependencies', {
    'install_time_s': round(time.time()-t0),
    'pyg_version': torch_geometric.__version__,
    'pytorch': torch.__version__,
})


## 2. Download CrisisMMD v2.0 Dataset

In [ ]:
import tarfile, shutil

DATA_DIR = 'data'
TARGET_DIR = os.path.join(DATA_DIR, 'CrisisMMD_v2.0')
TAR_PATH = os.path.join(DATA_DIR, 'CrisisMMD_v2.0.tar.gz')
URL = 'https://crisisnlp.qcri.org/data/crisismmd/CrisisMMD_v2.0.tar.gz'

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.isdir(TARGET_DIR):
    # Download with wget (10-50x faster than Python requests on Colab)
    if not os.path.exists(TAR_PATH) or os.path.getsize(TAR_PATH) < 1.8e9:
        print('Downloading dataset with wget...')
        !wget -q --show-progress -c -O {TAR_PATH} {URL}
    else:
        print(f'Archive exists: {os.path.getsize(TAR_PATH)/1e9:.2f} GB')

    # Extract with system tar (faster than Python tarfile module)
    print('Extracting...')
    !cd {DATA_DIR} && tar xzf CrisisMMD_v2.0.tar.gz
    print('Extraction complete!')
else:
    print(f'Dataset already exists at {TARGET_DIR}')

assert os.path.isdir(TARGET_DIR), f'Dataset not found: {TARGET_DIR}'
print(f'Dataset ready: {TARGET_DIR}')

# -- Log dataset --
import glob as _glob
_n_images = len(_glob.glob(os.path.join(TARGET_DIR, '**', '*.jpg'), recursive=True))
_n_images += len(_glob.glob(os.path.join(TARGET_DIR, '**', '*.png'), recursive=True))
LOG.log_stage('Dataset', {
    'path': TARGET_DIR,
    'num_images': _n_images,
    'size_gb': round(sum(os.path.getsize(os.path.join(dp,f)) for dp,dn,fn in os.walk(TARGET_DIR) for f in fn)/1e9, 2)
})


## 3. Feature Extraction (Cached)

Extract features using **three** extractors, cached to `.pkl`:  
- **MaxViT** (image) + **MPNet** (text) — repo's default  
- **CLIP ViT-B/32** (image) + **CLIP Text** — paper's baseline  

Subsequent runs use cache (~2s vs ~8min per extraction).

In [ ]:
import pickle

CACHE_DIR = 'cached_features'
os.makedirs(CACHE_DIR, exist_ok=True)

# ── Inject CLIP functions into feature_extraction.py ──
CLIP_MARKER = '# === CLIP_SUPPORT_V2 ==='
with open('feature_extraction.py', 'r') as f:
    fe_code = f.read()

if CLIP_MARKER not in fe_code:
    # Remove old CLIP injection if present
    if '# === CLIP_SUPPORT_ADDED ===' in fe_code:
        fe_code = fe_code.split('# === CLIP_SUPPORT_ADDED ===')[0]
        print('Removed old CLIP injection (upgrading to v2)')
    CLIP_INJECT = '''
# === CLIP_SUPPORT_V2 ===
# CLIP feature extraction using sentence-transformers (Multilingual CLIP)
# Paper: "multilingual version of CLIP based on ViT"
# Model: clip-ViT-B-32-multilingual-v1 (supports 50+ languages)
# Adds: clip_image_features() and clip_text_features()

from sentence_transformers import SentenceTransformer as _ST_CLIP
from PIL import Image as _PILImage

_CLIP_IMAGE_MODEL = "clip-ViT-B-32"  # vision encoder (encodes images)
_CLIP_TEXT_MODEL  = "clip-ViT-B-32-multilingual-v1"  # multilingual text encoder

def clip_image_features(gpu, sample=False):
    """Extract image features using multilingual CLIP ViT-B/32 vision encoder.
    Uses sentence-transformers multilingual CLIP as described in the paper.
    Returns list of 3 DataFrames [train, dev, test] with columns:
        image_files, text, embeddings, labels
    Embedding dim: 512
    """
    dfs = [None, None, None]

    if torch.cuda.is_available():
        dev = "cuda:" + str(gpu)
    else:
        dev = "cpu"

    # Load multilingual CLIP model (shared embedding space for images and text)
    model = _ST_CLIP(_CLIP_IMAGE_MODEL, cache_folder=".")
    model = model.to(dev)

    batch_size = 64

    for ix, split in enumerate(("train", "dev", "test")):
        data = [obj for obj in jsonlines.open(join(DATAPATH, split + ".jsonl"))]

        image_files = []
        text = []
        labels = []

        for row in tqdm.tqdm(data, desc=f"CLIP-img loading {split}"):
            image_files.append(join(IMAGEPATH, row['image']))
            text.append(row['text'])
            labels.append(row['label'])

        all_embeddings = []

        for i in tqdm.tqdm(range(0, len(image_files), batch_size),
                           desc=f"CLIP-img encoding {split}"):
            batch_files = image_files[i:i+batch_size]
            batch_images = []

            for img_path in batch_files:
                try:
                    img = _PILImage.open(img_path).convert("RGB")
                    batch_images.append(img)
                except Exception as e:
                    print(f"  Warning: Cannot load {img_path}: {e}")
                    batch_images.append(_PILImage.new("RGB", (224, 224), (128, 128, 128)))

            if not batch_images:
                continue

            # SentenceTransformer.encode handles PIL images directly
            embs = model.encode(batch_images, batch_size=len(batch_images),
                               show_progress_bar=False, convert_to_numpy=True)
            # L2 normalize
            norms = np.linalg.norm(embs, axis=1, keepdims=True)
            norms[norms == 0] = 1
            embs = embs / norms

            all_embeddings.extend(embs.tolist())

        labels_arr = [(1, 0) if l == 'not_informative' else (0, 1) for l in labels]
        labels_arr = np.array(labels_arr)

        df = pd.DataFrame({
            "image_files": image_files,
            "text": text,
            "embeddings": [e for e in all_embeddings],
            "labels": np.argmax(labels_arr, axis=1)
        })

        dfs[ix] = df
        print(f"  CLIP image features [{split}]: {len(df)} samples, dim={len(all_embeddings[0]) if all_embeddings else '?'}")

    return dfs


def clip_text_features(gpu, sample=False):
    """Extract text features using multilingual CLIP text encoder.
    Uses sentence-transformers multilingual CLIP as described in the paper.
    Supports 50+ languages for non-English crisis tweets.
    Returns list of 3 DataFrames [train, dev, test] with columns:
        image_files, text, clean_tex, embeddings, labels
    Embedding dim: 512
    """
    dfs = [None, None, None]

    if torch.cuda.is_available():
        dev = "cuda:" + str(gpu)
    else:
        dev = "cpu"

    # Load multilingual CLIP model (text encoder supports 50+ languages)
    model = _ST_CLIP(_CLIP_TEXT_MODEL, cache_folder=".")
    model = model.to(dev)

    batch_size = 128

    for ix, split in enumerate(("train", "dev", "test")):
        data = [obj for obj in jsonlines.open(join(DATAPATH, split + ".jsonl"))]

        image_files = []
        text = []
        clean_text = []
        labels = []

        for row in tqdm.tqdm(data, desc=f"CLIP-txt loading {split}"):
            image_files.append(join(IMAGEPATH, row['image']))
            text.append(row['text'])
            clean_text.append(preprocess.pre_process(row['text'],
                              keep_hashtag=True, keep_special_symbols=True))
            labels.append(row['label'])

        # SentenceTransformer.encode handles text natively (multilingual)
        all_embeddings = model.encode(clean_text, batch_size=batch_size,
                                      show_progress_bar=True, device=dev,
                                      convert_to_numpy=True)
        # L2 normalize
        norms = np.linalg.norm(all_embeddings, axis=1, keepdims=True)
        norms[norms == 0] = 1
        all_embeddings = all_embeddings / norms
        all_embeddings = all_embeddings.tolist()

        labels_arr = [(1, 0) if l == 'not_informative' else (0, 1) for l in labels]
        labels_arr = np.array(labels_arr)

        df = pd.DataFrame({
            "image_files": image_files,
            "text": text,
            "clean_tex": clean_text,
            "embeddings": [e for e in all_embeddings],
            "labels": np.argmax(labels_arr, axis=1)
        })

        dfs[ix] = df
        print(f"  CLIP text features [{split}]: {len(df)} samples, dim={len(all_embeddings[0]) if all_embeddings else '?'}")

    return dfs
'''
    with open('feature_extraction.py', 'a') as f:
        f.write(CLIP_INJECT)
    print('✅ Injected CLIP functions into feature_extraction.py')
else:
    print('CLIP functions already present in feature_extraction.py')

# ── Patch feature_fusion.py for CLIP support ──
CLIP_FF_MARKER = 'clip_image'
with open('feature_fusion.py', 'r') as f:
    ff_code = f.read()

if CLIP_FF_MARKER not in ff_code:
    # Add clip_image to imageft choices
    ff_code = ff_code.replace(
        "choices=['maxvit', 'mobilenet', 'mobilevit']",
        "choices=['maxvit', 'mobilenet', 'mobilevit', 'clip_image']")
    # Add clip_text to textft choices
    ff_code = ff_code.replace(
        "choices=['mpnet']",
        "choices=['mpnet', 'clip_text']")
    # Add dispatch for clip_text
    ff_code = ff_code.replace(
        'if args_cl.textft == "mpnet":',
        'if args_cl.textft == "clip_text":\n'
        '    [df_text_train, df_text_dev, df_text_test] = clip_text_features(dev_id, args_cl.sample)\n'
        'elif args_cl.textft == "mpnet":')
    # Add dispatch for clip_image
    ff_code = ff_code.replace(
        'elif args_cl.imageft == "maxvit":',
        'elif args_cl.imageft == "clip_image":\n'
        '    [df_image_train, df_image_dev, df_image_test] = clip_image_features(dev_id, args_cl.sample)\n'
        'elif args_cl.imageft == "maxvit":')
    with open('feature_fusion.py', 'w') as f:
        f.write(ff_code)
    print('✅ Patched feature_fusion.py with CLIP dispatch')
else:
    print('feature_fusion.py already patched for CLIP')

# ── Extract & Cache all features ──
from importlib import reload
import feature_extraction
reload(feature_extraction)
from feature_extraction import (
    mpnet_features, maxvit_features,
    clip_image_features, clip_text_features
)

# ── Smart extraction: only what's needed ──
# Read EXPERIMENT_MODE early to skip unnecessary extractors
_MODE = os.environ.get('EXPERIMENT_MODE', 'maxvit_mpnet')  # default

if _MODE.startswith('clip'):
    EXTRACTORS = [
        ('clip_image_features.pkl', 'CLIP (image)',  clip_image_features),
        ('clip_text_features.pkl',  'CLIP (text)',   clip_text_features),
    ]
    print('Mode:', _MODE, '-> extracting CLIP features only')
elif _MODE == 'maxvit_mpnet':
    EXTRACTORS = [
        ('mpnet_features.pkl',      'MPNet (text)',   mpnet_features),
        ('maxvit_features.pkl',     'MaxViT (image)', maxvit_features),
    ]
    print('Mode:', _MODE, '-> extracting MaxViT+MPNet only')
else:
    EXTRACTORS = [
        ('mpnet_features.pkl',      'MPNet (text)',     mpnet_features),
        ('maxvit_features.pkl',     'MaxViT (image)',   maxvit_features),
        ('clip_image_features.pkl', 'CLIP (image)',     clip_image_features),
        ('clip_text_features.pkl',  'CLIP (text)',      clip_text_features),
    ]
    print('Mode:', _MODE, '-> extracting ALL features')

for cache_name, label, fn in EXTRACTORS:
    cache_path = os.path.join(CACHE_DIR, cache_name)
    if not os.path.exists(cache_path):
        print(f'Extracting {label}...')
        t0 = time.time()
        dfs = fn(0, sample=False)
        with open(cache_path, 'wb') as f:
            pickle.dump(dfs, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f'  Cached: {os.path.getsize(cache_path)/1e6:.1f} MB ({time.time()-t0:.0f}s)')
        # Free memory
        del dfs
        import gc; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    else:
        print(f'{label} cached: {os.path.getsize(cache_path)/1e6:.1f} MB')

print('\nFeatures ready for mode:', _MODE)

# -- Log features --
import glob as _glob
_caches = _glob.glob(os.path.join(CACHE_DIR, '*.pkl'))
LOG.log_stage('Feature Extraction', {
    'mode': _MODE,
    'cached_files': [os.path.basename(c) for c in _caches],
    'total_cache_mb': round(sum(os.path.getsize(c) for c in _caches)/1e6, 1)
})


## 4. Patch feature_extraction.py for Cache Loading

Append cache-loading wrappers so `feature_fusion.py` (called via subprocess)  
automatically loads from `.pkl` files instead of re-extracting.

In [ ]:
PATCH_CODE = '''
# === CACHE_PATCH_APPLIED ===
import pickle as _pkl
import os as _os
import time as _time
_CACHE_DIR = 'cached_features'

try:
    _orig_mpnet
except NameError:
    _orig_mpnet = mpnet_features
    _orig_maxvit = maxvit_features
    _orig_clip_img = clip_image_features
    _orig_clip_txt = clip_text_features

def mpnet_features(*args, **kwargs):
    _c = _os.path.join(_CACHE_DIR, 'mpnet_features.pkl')
    if _os.path.exists(_c):
        _s = _time.time()
        with open(_c, 'rb') as _f: _r = _pkl.load(_f)
        print(f'Loaded cached MPNet text features ({_time.time()-_s:.1f}s)')
        return _r
    return _orig_mpnet(*args, **kwargs)

def maxvit_features(*args, **kwargs):
    _c = _os.path.join(_CACHE_DIR, 'maxvit_features.pkl')
    if _os.path.exists(_c):
        _s = _time.time()
        with open(_c, 'rb') as _f: _r = _pkl.load(_f)
        print(f'Loaded cached MaxViT image features ({_time.time()-_s:.1f}s)')
        return _r
    return _orig_maxvit(*args, **kwargs)

def clip_image_features(*args, **kwargs):
    _c = _os.path.join(_CACHE_DIR, 'clip_image_features.pkl')
    if _os.path.exists(_c):
        _s = _time.time()
        with open(_c, 'rb') as _f: _r = _pkl.load(_f)
        print(f'Loaded cached CLIP image features ({_time.time()-_s:.1f}s)')
        return _r
    return _orig_clip_img(*args, **kwargs)

def clip_text_features(*args, **kwargs):
    _c = _os.path.join(_CACHE_DIR, 'clip_text_features.pkl')
    if _os.path.exists(_c):
        _s = _time.time()
        with open(_c, 'rb') as _f: _r = _pkl.load(_f)
        print(f'Loaded cached CLIP text features ({_time.time()-_s:.1f}s)')
        return _r
    return _orig_clip_txt(*args, **kwargs)
'''

MARKER = '# === CACHE_PATCH_APPLIED ==='
with open('feature_extraction.py', 'r') as f:
    orig = f.read()

if MARKER not in orig:
    with open('feature_extraction.py', 'w') as f:
        f.write(orig + '\n' + PATCH_CODE)
    print('Patched feature_extraction.py with cache loading (MaxViT, MPNet, CLIP)')
else:
    print('Already patched')

## 5. Run Experiments

Configuration matches **`exp_5.sh`** from the original repo:  
- **Fusion:** Late (no reduction) — paper's best config  
- **Image:** MaxViT | **Text:** MPNet  
- **GNN:** `sage-2l-norm-res` (2-layer GraphSAGE, BatchNorm, Residual, 1024 hidden)  
- **Epochs:** 2000 | **Early Stopping:** 300 | **Best Model:** harmonic mean  
- **Labeled sets:** 50, 100, 250, 500 × 10 splits = **40 runs total**  

Each run takes ~5-10 min on T4 GPU ≈ **3-7 hours total**.

In [ ]:
# ═══════════════════════════════════════════════════════
# 5a. Configuration — Select experiment mode
# ═══════════════════════════════════════════════════════

# ┌─────────────────────────────────────────────────┐
# │ EXPERIMENT MODE: Choose one                     │
# │  'maxvit_mpnet'  → exp_5 (paper's best config) │
# │  'clip_pca'      → CLIP + PCA (paper baseline) │
# │  'clip_late'     → CLIP late fusion (no PCA)   │
# └─────────────────────────────────────────────────┘
EXPERIMENT_MODE = 'maxvit_mpnet'  # <- Change this to switch experiments

EPOCHS = 2000
EARLY_STOP = 300

# Base hyperparameters (shared across all modes)
BASE_CFG = dict(
    lr       = 1e-5,
    wd       = 1e-3,
    l2       = 1e-2,
    do       = 0.5,
    knn      = 16,
    frac     = 0.4,
    arch     = 'sage-2l-norm-res',
)

# Mode-specific overrides
MODE_CONFIGS = {
    'maxvit_mpnet': dict(
        img='maxvit', txt='mpnet',
        fusion='late', reduction=None,
        exp_id=5, desc='MaxViT+MPNet Late (paper best)'
    ),
    'clip_pca': dict(
        img='clip_image', txt='clip_text',
        fusion='late', reduction='pca', pca_red=256,
        exp_id=100, desc='CLIP+PCA Late Fusion'
    ),
    'clip_late': dict(
        img='clip_image', txt='clip_text',
        fusion='late', reduction=None,
        exp_id=101, desc='CLIP Late Fusion (no reduction)'
    ),
}

mode = MODE_CONFIGS[EXPERIMENT_MODE]
CFG = {**BASE_CFG, **mode}
EXP_ID = CFG.pop('exp_id')
EXP_DESC = CFG.pop('desc')

# Labeled set sizes & splits
LABELED_SIZES = [50, 100, 250, 500]
NUM_SPLITS = 10

RUNS = []
for n in LABELED_SIZES:
    for s in range(NUM_SPLITS):
        RUNS.append({'n': n, 'split': f'{n}_s{s}', 's': s})

print(f'Experiment: {EXP_DESC}')
print(f'  Mode: {EXPERIMENT_MODE} | EXP_ID: {EXP_ID}')
print(f'  Fusion: {CFG["fusion"]} | Arch: {CFG["arch"]}')
print(f'  Features: {CFG["img"]} (image) + {CFG["txt"]} (text)')
if CFG.get('reduction'):
    print(f'  Reduction: {CFG["reduction"]} (dim={CFG.get("pca_red", "N/A")})')
else:
    print(f'  Reduction: None')
print(f'  Epochs: {EPOCHS} | Early Stop: {EARLY_STOP}')
print(f'  Total runs: {len(RUNS)} ({len(LABELED_SIZES)} sizes x {NUM_SPLITS} splits)')
# -- Log experiment config --
LOG.set_meta('experiment_mode', EXPERIMENT_MODE)
LOG.log_stage('Experiment Config', {
    'mode': EXPERIMENT_MODE,
    'fusion': CFG.get('fusion'),
    'arch': CFG.get('arch'),
    'img_features': CFG.get('img'),
    'txt_features': CFG.get('txt'),
    'reduction': CFG.get('reduction'),
    'epochs': EPOCHS,
    'early_stop': EARLY_STOP,
    'labeled_sizes': LABELED_SIZES,
    'num_splits': NUM_SPLITS,
    'total_runs': len(RUNS)
})


In [ ]:
# ═══════════════════════════════════════════════════════
# 5b. Clean old results (optional — only for fresh run)
# ═══════════════════════════════════════════════════════
import glob

# Uncomment the next 3 lines to clear previous results:
# if os.path.isdir('results'):
#     shutil.rmtree('results')
#     print('Removed old results')

existing = glob.glob('results/**/*.json', recursive=True)
print(f'Existing result files: {len(existing)}')

In [ ]:
# ═══════════════════════════════════════════════════════
# 5c. Run all experiments via subprocess
#     (supports MaxViT/MPNet, CLIP, PCA, etc.)
# ═══════════════════════════════════════════════════════
import subprocess

IP = './data/CrisisMMD_v2.0/'
RUN_ID = 0

total_t0 = time.time()
completed = 0
failed = 0

for i, r in enumerate(RUNS, 1):
    # Skip if result already exists
    result_path = f'results/CrisisMMD/gnn/{EXP_ID}/{r["n"]}_s{r["s"]}_{RUN_ID}.json'
    if os.path.exists(result_path):
        print(f'[{i}/{len(RUNS)}] SKIP {r["split"]} (result exists)')
        completed += 1
        continue

    # Build command
    cmd = (
        f'"{sys.executable}" feature_fusion.py'
        f' --gpu_id 0'
        f' --epochs {EPOCHS}'
        f' --lr {CFG["lr"]}'
        f' --weight_decay {CFG["wd"]}'
        f' --n_neigh_train {CFG["knn"]}'
        f' --n_neigh_full {CFG["knn"]}'
        f' --lbl_train_frac {CFG["frac"]}'
        f' --imagepath {IP}'
        f' --datasplit {r["split"]}'
        f' --reg l2'
        f' --l2_lambda {CFG["l2"]}'
        f' --exp_id {EXP_ID}'
        f' --dropout {CFG["do"]}'
        f' --arch {CFG["arch"]}'
        f' --loss nll'
        f' --shuffle_split'
        f' --imageft {CFG["img"]}'
        f' --textft {CFG["txt"]}'
        f' --fusion {CFG["fusion"]}'
        f' --early_stopping {EARLY_STOP}'
        f' --best_model best_hm'
        f' --run_id {RUN_ID}'
    )

    # Add optional reduction flags
    if CFG.get('reduction'):
        cmd += f' --reduction {CFG["reduction"]}'
        if CFG['reduction'] == 'pca' and CFG.get('pca_red'):
            cmd += f' --pca_red {CFG["pca_red"]}'
        elif CFG['reduction'] == 'umap' and CFG.get('umap_red'):
            cmd += f' --umap_red {CFG["umap_red"]}'
        elif CFG['reduction'] == 'autoenc' and CFG.get('autoenc'):
            cmd += f' --autoenc {CFG["autoenc"]}'

    print(f'\n[{i}/{len(RUNS)}] Running: {r["split"]}')
    t0 = time.time()
    try:
        p = subprocess.run(cmd, shell=True, timeout=7200)
        ok = (p.returncode == 0)
    except subprocess.TimeoutExpired:
        print(f'  TIMEOUT after 2h'); ok = False
    except Exception as e:
        print(f'  ERROR: {e}'); ok = False

    elapsed = time.time() - t0
    status = 'OK' if ok else 'FAIL'
    print(f'  -> {status} ({elapsed:.0f}s)')
    if ok:
        completed += 1
    else:
        failed += 1

total_time = time.time() - total_t0
print(f'\n{"="*50}')
print(f'ALL RUNS COMPLETE ({EXP_DESC})')
print(f'  Completed: {completed}/{len(RUNS)}')
print(f'  Failed: {failed}')
print(f'  Total time: {total_time/3600:.1f}h')

## 6. Results — Aggregation & Paper Comparison

In [ ]:
# ═══════════════════════════════════════════════════════
# 6a. Load & aggregate results
# ═══════════════════════════════════════════════════════
import json, glob, numpy as np
from collections import defaultdict
from IPython.display import display, HTML

results = []
for f in glob.glob('results/**/*.json', recursive=True):
    try:
        with open(f) as fp:
            d = json.load(fp)
        d['_file'] = f
        results.append(d)
    except:
        pass

print(f'Loaded {len(results)} result files')

# Group by labeled-set size
groups = defaultdict(list)
for r in results:
    fname = os.path.basename(r['_file'])
    try:
        size = int(fname.split('_')[0])
        groups[size].append(r)
    except:
        pass

# Paper comparison values (Table V)
# Source: Paper's reported F1-Score for exp_5 (late fusion, MaxViT+MPNet)
paper_f1 = {
    50:  74.8,
    100: 76.3,
    250: 77.1,
    # 500: Not explicitly reported for all configs in paper
}
paper_std = {
    50:  2.0,
    100: 0.7,
    250: 0.4,
}

# Sirbu et al. baseline (from paper Table III)
sirbu_f1 = {
    50:  62.8,
    100: 66.9,
    250: 71.9,
}

# Build comparison table
h = '<h3>Aggregated Results (Mean ± Std over splits)</h3>'
h += '<table border=1 cellpadding=6 style="border-collapse:collapse;">'
h += '<tr style="background:#f0f0f0;">'
h += '<th>Labeled</th><th># Splits</th>'
h += '<th>Our F1 (%)</th><th>Paper F1 (%)</th>'
h += '<th>Sirbu F1 (%)</th><th>Gap vs Paper</th></tr>'

for size in sorted(groups.keys()):
    runs = groups[size]
    f1s = [r.get('f1_test', 0) * 100 for r in runs]
    n = len(runs)
    f1_mean, f1_std = np.mean(f1s), np.std(f1s)

    pf1 = paper_f1.get(size, '—')
    pstd = paper_std.get(size, '')
    sf1 = sirbu_f1.get(size, '—')

    gap = f'{f1_mean - pf1:+.1f}' if isinstance(pf1, (int, float)) else '—'
    pf1_str = f'{pf1}±{pstd}' if pstd else str(pf1)

    color = '#e8f5e9' if isinstance(pf1, (int,float)) and f1_mean >= pf1 else '#fff3e0'
    h += f'<tr style="background:{color};">'
    h += f'<td><b>{size}</b></td><td>{n}</td>'
    h += f'<td>{f1_mean:.1f} ± {f1_std:.1f}</td>'
    h += f'<td>{pf1_str}</td>'
    h += f'<td>{sf1}</td>'
    h += f'<td>{gap}</td></tr>'

h += '</table>'

# Individual runs (collapsible)
h += '<br><details><summary>Click to show individual runs</summary>'
h += '<table border=1 cellpadding=4 style="border-collapse:collapse;">'
h += '<tr><th>File</th><th>F1 Test (%)</th><th>BACC Test (%)</th></tr>'
for r in sorted(results, key=lambda x: x['_file']):
    f1 = r.get('f1_test', 0) * 100
    bacc = r.get('bacc_test', 0) * 100
    h += f'<tr><td>{os.path.basename(r["_file"])}</td>'
    h += f'<td>{f1:.2f}</td><td>{bacc:.2f}</td></tr>'
h += '</table></details>'

if results:
    display(HTML(h))
else:
    print('No result files found. Run experiments first.')

In [ ]:
# ═══════════════════════════════════════════════════════
# 6b. Comparison Plot
# ═══════════════════════════════════════════════════════
import matplotlib.pyplot as plt

S = sorted(groups.keys()) if groups else [50, 100, 250, 500]
S_paper = [s for s in S if s in paper_f1]
S_sirbu = [s for s in S if s in sirbu_f1]

fig, ax = plt.subplots(figsize=(10, 6))

# Paper results
if S_paper:
    ax.errorbar(
        S_paper, [paper_f1[s] for s in S_paper],
        yerr=[paper_std.get(s, 0) for s in S_paper],
        fmt='g-o', lw=2, ms=8, capsize=4,
        label='Paper: GNN Late Fusion'
    )

# Sirbu baseline
if S_sirbu:
    ax.plot(S_sirbu, [sirbu_f1[s] for s in S_sirbu],
            'r--s', lw=1.5, ms=7, label='Sirbu et al. (baseline)')

# Our reproduction
if groups:
    our_S, our_means, our_stds = [], [], []
    for s in sorted(groups.keys()):
        f1s = [r.get('f1_test', 0) * 100 for r in groups[s]]
        our_S.append(s)
        our_means.append(np.mean(f1s))
        our_stds.append(np.std(f1s))
    ax.errorbar(
        our_S, our_means, yerr=our_stds,
        fmt='b-D', lw=2, ms=8, capsize=4,
        label='Our Reproduction'
    )

ax.set_xlabel('Number of Labeled Samples', fontsize=12)
ax.set_ylabel('F1-Score (%)', fontsize=12)
ax.set_title('Few-Shot CrisisMMD Classification — Paper vs Reproduction', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks([50, 100, 250, 500])
ax.set_ylim(55, 85)
plt.tight_layout()
plt.savefig('reproduction_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved: reproduction_comparison.png')

## 7. Experiment Report

Auto-generated report with timing, config, and results from all stages.  
Files: `experiment_report_{mode}.json` + `.md` — download for your report.

In [ ]:
# ===============================================================
# 7. Save Full Experiment Report (auto-download)
# ===============================================================

import glob, datetime

# -- Collect result files --
result_files = sorted(glob.glob('results/**/*.json', recursive=True))
LOG.log_stage('Results Summary', {
    'num_result_files': len(result_files),
    'experiment_mode': EXPERIMENT_MODE,
})

# -- Save JSON report --
json_path = f'experiment_report_maxvit_mpnet.json'
report = LOG.save(json_path)

# -- Generate detailed Markdown report --
md_lines = []
md_lines.append('# Experiment Report')
md_lines.append(f'**Generated:** {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
md_lines.append(f'**Total runtime:** {report["total_time_s"]/60:.1f} minutes')
md_lines.append('')

# Metadata section
md_lines.append('## Experiment Configuration')
for k, v in report['metadata'].items():
    md_lines.append(f'- **{k}:** {v}')
md_lines.append('')

# Stages section
md_lines.append('## Execution Stages')
md_lines.append('| Stage | Time (min) | Details |')
md_lines.append('|-------|-----------|---------|')
for s in report['stages']:
    details = ', '.join(f'{k}={v}' for k,v in s.items() if k not in ('stage','timestamp','elapsed_s'))
    md_lines.append(f'| {s["stage"]} | {s["elapsed_s"]/60:.1f} | {details[:80]} |')
md_lines.append('')

# Results table (if aggregated results exist)
try:
    md_lines.append('## Results')
    md_lines.append('| Labeled | Splits | Our F1 (%) | Paper F1 (%) |')
    md_lines.append('|---------|--------|-----------|-------------|')
    for _, row in agg.iterrows():
        n = int(row['n_labeled'])
        f1 = f'{row["f1_mean"]*100:.1f} +/- {row["f1_std"]*100:.1f}'
        pf1 = paper_f1.get(n, '--')
        md_lines.append(f'| {n} | {int(row["n_splits"])} | {f1} | {pf1} |')
    md_lines.append('')
except:
    md_lines.append('*(Results not available - run Cell 6 first)*')
    md_lines.append('')

md_text = chr(10).join(md_lines)
md_path = f'experiment_report_maxvit_mpnet.md'
with open(md_path, 'w', encoding='utf-8') as f:
    f.write(md_text)
print(f'Markdown report saved: {md_path}')

# -- Display in notebook --
from IPython.display import Markdown, display
display(Markdown(md_text))

# -- Auto download --
try:
    from google.colab import files
    files.download(json_path)
    files.download(md_path)
    # Download individual result files too
    for rf in result_files[-5:]:
        files.download(rf)
    print(f'\nAuto-downloaded {2 + min(5, len(result_files))} files!')
except:
    print('Files saved. Download from Colab file browser (left panel).')


---

## Notes

### Experiment Configuration Reference

This notebook supports **multiple experiment modes** via `EXPERIMENT_MODE` in Cell 5a:

| Mode | Image | Text | Fusion | Reduction | EXP_ID | Description |
|---|---|---|---|---|---|---|
| `maxvit_mpnet` | MaxViT | MPNet | Late | None | 5 | Repo's best config |
| `clip_pca` | CLIP Multilingual | CLIP Multilingual | Late | PCA-256 | 100 | Paper's CLIP + PCA |
| `clip_late` | CLIP Multilingual | CLIP Multilingual | Late | None | 101 | CLIP without reduction |

### CLIP Implementation Details

Paper: *"multilingual version of CLIP based on ViT, pre-trained on WIT"*

- **Model:** `clip-ViT-B-32-multilingual-v1` (via `sentence-transformers`)
- **Image embedding:** 512-dim (L2-normalized) via `model.encode(images)`
- **Text embedding:** 512-dim (L2-normalized) via `model.encode(texts)` -- supports 50+ languages
- **Text preprocessing:** Same as MPNet (via `preprocess.py`)
- **PCA reduction:** 256 components (when `reduction='pca'`)

### Original Repo Experiments (All MaxViT + MPNet)

| Exp | Fusion | Reduction | Notes |
|---|---|---|---|
| 1 | Late | Autoencoder | |
| 3 | Late | PCA (256) | |
| 5 | Late | None | **Paper's best** |
| 6 | Late Linear | None | |
| 10 | Middle | PCA (256) | |
| 11 | Early | None | |

### Feature Caching

All feature extractors (MaxViT, MPNet, CLIP image, CLIP text) are cached to `cached_features/`.  
This saves significant time across multiple experiment runs.

### Bugs Fixed from Original Repo

- `--n_neigh_test` → `--n_neigh_full` (correct CLI argument)
- `python3` → `sys.executable` (Windows/Colab compatibility)
- Safe tar extraction (handles path conflicts)
- Added CLIP support (`clip_image`, `clip_text`) not in original repo